<a href="https://colab.research.google.com/github/MurtonN/Federated-Learning-Research-UW-Platteville/blob/main/Intrusion_Detection_CICIDS2017.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler, StandardScaler
import seaborn as sns
import matplotlib.pyplot as plt
import warnings

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report
from sklearn.metrics import confusion_matrix

warnings.filterwarnings('ignore')

In [22]:
data1 = pd.read_csv("Wednesday-workingHours.pcap_ISCX.csv") # loads and reads the dataset

In [ ]:
data1.columns = data1.columns.str.strip() # strips data of whitespace
data1["Label"] = (data1["Label"] == "BENIGN").astype(int) # converts object to int
data1.rename(columns={data1.columns[55]: "Fwd Header Length"}, inplace=True) # renames all weird names
data1.rename(columns={data1.columns[66]: "Init Win Bytes Fwd"}, inplace=True)
data1.rename(columns={data1.columns[67]: "Init Win Bytes Bwd"}, inplace=True)
data1.rename(columns={data1.columns[68]: "Act Data Fwd Packets"}, inplace=True)
data1.rename(columns={data1.columns[69]: "Min Seg Fwd Size"}, inplace=True)
data1 = data1.drop_duplicates() # removes duplicate values
data1.replace([np.inf, -np.inf], np.nan, inplace=True) # replaces infinite values with missing values
data1_cleaned = data1.dropna() # drops all rows with missing values

In [ ]:
data1.head()

,Destination Port,Flow Duration,Total Fwd Packets,Total Backward Packets,Total Length of Fwd Packets,Total Length of Bwd Packets,Fwd Packet Length Max,Fwd Packet Length Min,Fwd Packet Length Mean,Fwd Packet Length Std,...,Min Seg Fwd Size,Active Mean,Active Std,Active Max,Active Min,Idle Mean,Idle Std,Idle Max,Idle Min,Label
0,80,38308,1,1,6,6,6,6,6.000000,0.000000,...,20,0.0,0.0,0,0,0.0,0.0,0,0,1
1,389,479,11,5,172,326,79,0,15.636364,31.449238,...,32,0.0,0.0,0,0,0.0,0.0,0,0,1
2,88,1095,10,6,3150,3150,1575,0,315.000000,632.561635,...,32,0.0,0.0,0,0,0.0,0.0,0,0,1
3,389,15206,17,12,3452,6660,1313,0,203.058823,425.778474,...,32,0.0,0.0,0,0,0.0,0.0,0,0,1
4,88,1092,9,6,3150,3152,1575,0,350.000000,694.509719,...,32,0.0,0.0,0,0,0.0,0.0,0,0,1


In [ ]:
Y = data1["Label"]
X = data1.drop("Label", axis=1)

In [ ]:
percentages = data1["Label"].value_counts(normalize=True) * 100
print(percentages)

Label
1    68.26232
0    31.73768
Name: proportion, dtype: float64


In [ ]:
X_train, X_test, Y_train, Y_test = train_test_split(
    X, Y,
    test_size=0.2,
    random_state=42
)

In [ ]:
regressor = RandomForestRegressor(
    n_estimators=25,
    random_state=42,
    oob_score=True
)

regressor.fit(X_train, Y_train)

RandomForestRegressor(n_estimators=25, oob_score=True, random_state=42)

In [ ]:
print("Out-of-Bag Score:", regressor.oob_score_)

Y_pred = regressor.predict(X_test)

mse = mean_squared_error(Y_test, Y_pred)
print("Mean Squared Error:", mse)

r2 = r2_score(Y_test, Y_pred)
print("R-squared:", r2)

Out-of-Bag Score: 0.9985796648863111
Mean Squared Error: 0.00030182692714374616
R-squared: 0.9986089185928934


In [ ]:
Y_pred = regressor.predict(X_test)
Y_pred_binary = (Y_pred > 0.5).astype(int) # Convert continuous predictions to binary
print(classification_report(Y_test, Y_pred_binary))
cm = confusion_matrix(Y_test, Y_pred_binary)
tn, fp, fn, tp = cm.ravel()
print(f"True Negatives: {tn}, False Positives: {fp}, False Negatives: {fn}, True Positives: {tp}")

              precision    recall  f1-score   support

           0       1.00      1.00      1.00     38860
           1       1.00      1.00      1.00     83239

    accuracy                           1.00    122099
   macro avg       1.00      1.00      1.00    122099
weighted avg       1.00      1.00      1.00    122099

True Negatives: 38843, False Positives: 17, False Negatives: 27, True Positives: 83212


In [ ]:
data1_cleaned.to_csv("Wednesday_workingHours_cleaned.csv", index=False)

In [ ]:
from google.colab import files
files.download("Wednesday_workingHours_cleaned.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>